# Chapter 3 lab — Build a hexapod from the three nouns

Companion to **[Chapter 3 — MuJoCo Without Tears](https://kamatechorg.github.io/robo-greeno-data-a/tutorial/03-mujoco-intuition/)**.

Chapter 3 says every MuJoCo robot is just **bodies + joints + actuators**, and that our hexapod has *18 joints and 18 actuators* (3 per leg × 6 legs). Here we generate that robot in code, confirm the counts, watch the timestep matter, and drive the legs with an open-loop tripod gait — then render it.

~4 minutes. **CPU is fine.**

## Step 1 — install MuJoCo (~1 min)

In [ ]:
!pip install -q mujoco mediapy
print("install OK")

## Step 2 — generate the hexapod MJCF in Python

Rather than hand-write 18 near-identical legs, we build the XML in a loop — exactly the “tree of bodies” Chapter 3 describes: a torso at the root with six legs branching off, each leg a coxa→femur→tibia chain of 3 hinge joints.

In [ ]:
import numpy as np

def leg(i, angle_deg):
    a = np.deg2rad(angle_deg)
    x, y = 0.12*np.cos(a), 0.12*np.sin(a)
    return f'''
    <body name="leg{i}_coxa" pos="{x:.3f} {y:.3f} 0" euler="0 0 {angle_deg}">
      <joint name="leg{i}_coxa"  type="hinge" axis="0 0 1" range="-45 45"/>
      <geom type="capsule" fromto="0 0 0  0.05 0 0" size="0.012"/>
      <body name="leg{i}_femur" pos="0.05 0 0">
        <joint name="leg{i}_femur" type="hinge" axis="0 1 0" range="-60 60"/>
        <geom type="capsule" fromto="0 0 0  0.07 0 0" size="0.011"/>
        <body name="leg{i}_tibia" pos="0.07 0 0">
          <joint name="leg{i}_tibia" type="hinge" axis="0 1 0" range="-90 30"/>
          <geom type="capsule" fromto="0 0 0  0.09 0 -0.02" size="0.010"/>
        </body>
      </body>
    </body>'''

# six legs every 60 degrees around the torso
legs_xml = "".join(leg(i+1, ang) for i, ang in enumerate(range(0, 360, 60)))
actuators = "".join(
    f'<position name="leg{i}_{seg}" joint="leg{i}_{seg}" kp="3"/>'
    for i in range(1, 7) for seg in ("coxa", "femur", "tibia"))

XML = f'''
<mujoco>
  <option gravity="0 0 -9.81" timestep="0.002"/>
  <worldbody>
    <light pos="0 0 3"/>
    <geom type="plane" size="3 3 0.1" rgba=".6 .7 .6 1"/>
    <body name="torso" pos="0 0 0.18">
      <freejoint/>
      <geom type="box" size="0.10 0.10 0.025" rgba=".2 .3 .8 1" mass="0.5"/>
      {legs_xml}
    </body>
  </worldbody>
  <actuator>{actuators}</actuator>
</mujoco>'''
print("generated XML, length:", len(XML), "chars")

## Step 3 — load it and confirm the counts from the chapter

In [ ]:
import mujoco
model = mujoco.MjModel.from_xml_string(XML)
data = mujoco.MjData(model)

hinge = sum(1 for j in range(model.njnt)
            if model.jnt_type[j] == mujoco.mjtJoint.mjJNT_HINGE)
print(f"hinge joints (leg DOF): {hinge}   <- chapter says 18")
print(f"actuators (servos):      {model.nu}   <- chapter says 18")
print(f"bodies (incl. torso):    {model.nbody}")
assert hinge == 18 and model.nu == 18, "counts should match the chapter!"
print("counts match the chapter. ✓")

## Step 4 — why the timestep is small

Chapter 3: *big steps mean a foot can travel through the floor before anyone notices.* The visible symptom of too-large a step is the integrator going **unstable** — positions blow up to huge/NaN values instead of settling. We let the robot fall under gravity at a safe 2 ms step and a reckless 50 ms step and compare.

In [ ]:
def settle(dt, steps=200):
    m = mujoco.MjModel.from_xml_string(XML)
    m.opt.timestep = dt
    d = mujoco.MjData(m)
    for _ in range(steps):
        mujoco.mj_step(m, d)
    return d.qpos[2]  # final torso height

for dt in (0.002, 0.05):
    z = settle(dt)
    stable = np.isfinite(z) and abs(z) < 5      # sane value, not exploded
    shown = f"{z:+.3f} m" if np.isfinite(z) else "NaN / inf"
    note = "stable" if stable else "UNSTABLE — integrator exploded"
    print(f"timestep {dt*1000:>4.0f} ms -> torso height {shown:>12}  ({note})")

## Step 5 — drive the 18 actuators and render (~1 min)

We command the femur and tibia joints with sine waves, 180° out of phase between the two tripod groups (the Chapter 2 grouping), and render. The point here is *the actuators move the joints and the contact solver keeps the feet on the ground* — **learning an actual forward gait is Chapter 4's job**, not hand-tuned sines.

In [ ]:
import os; os.environ.setdefault("MUJOCO_GL", "egl")
import mediapy as media

model = mujoco.MjModel.from_xml_string(XML)
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=300, width=400)

tripod_A = {1, 4, 5}   # legs that step together (see Chapter 2 lab)
aid = lambda leg, seg: model.actuator(f"leg{leg}_{seg}").id

frames, stable = [], True
for t in range(1500):
    phase = 2*np.pi * t * model.opt.timestep * 1.0   # 1 Hz stride
    for leg in range(1, 7):
        ph = phase if leg in tripod_A else phase + np.pi
        data.ctrl[aid(leg, "femur")] = 0.4 * np.sin(ph)
        data.ctrl[aid(leg, "tibia")] = 0.3 * np.sin(ph)
    mujoco.mj_step(model, data)
    if not np.all(np.isfinite(data.qpos)):
        stable = False; break
    if t % 10 == 0:
        renderer.update_scene(data)
        frames.append(renderer.render())

print(f"sim stayed numerically stable: {stable}")
print(f"active foot/ground contacts at end: {data.ncon}")
media.show_video(frames, fps=30)

!!! abstract "What you just built"
    - A complete hexapod is **bodies + joints + actuators** — 18 hinges, 18 servos, generated in a `for` loop, not hand-drawn (the counts match the chapter exactly).
    - A 2 ms timestep stays stable; a 50 ms step makes the integrator explode — exactly the speed-vs-accuracy trade-off the chapter describes.
    - The 18 actuators drive the joints and MuJoCo's **contact solver** (the “Co”) keeps the feet on the ground — watch `data.ncon` report live contacts.
    - Open-loop sine waves just wiggle the legs. *Learning* a gait that actually walks is Chapter 4's job.

**Try it:** change the stride frequency or the tripod grouping and re-render. Next: **[Chapter 4 — Reinforcement Learning, Intuitively](https://kamatechorg.github.io/robo-greeno-data-a/tutorial/04-rl-intuition/)**.